 # Nyaya-Sahayak — Query Analytics Dashboard
 # Spark SQL analytics on the `query_logs` Delta table.
# Run this after users have interacted with the app.
 # | Metric | SQL |
 |--------|-----|
 | Domain distribution | GROUP BY domain_detected |
 | Language breakdown | GROUP BY user_lang |
 | Avg response time | AVG(response_time_ms) |
 | Top cited sections | JSON parse sections_cited |
 | Query volume over time | GROUP BY date |


## 0. Config

In [0]:

CATALOG = "workspace"
SCHEMA = "default"
TABLE = f"{CATALOG}.{SCHEMA}.query_logs"

# Ensure the table exists (create empty if not)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        query_id STRING,
        timestamp STRING,
        user_lang STRING,
        query_text STRING,
        query_en STRING,
        domain_detected STRING,
        sections_cited STRING,
        response_time_ms INT,
        model_used STRING,
        retrieval_backend STRING
    )
    USING DELTA
""")
print(f"✅ Table {TABLE} ready")

row_count = spark.table(TABLE).count()
print(f"📊 Total logged queries: {row_count}")

## 1. Legal Domain Distribution
 Which legal domains are most queried?

In [0]:

%sql
SELECT 
    domain_detected AS legal_domain,
    COUNT(*) AS query_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM workspace.default.query_logs
WHERE domain_detected IS NOT NULL AND domain_detected != ''
GROUP BY domain_detected
ORDER BY query_count DESC


## 2. Language Distribution
What languages are users asking in?

In [0]:
%sql
SELECT 
    user_lang AS language,
    COUNT(*) AS query_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM workspace.default.query_logs
WHERE user_lang IS NOT NULL
GROUP BY user_lang
ORDER BY query_count DESC


## 3. Response Time Analytics
How fast is the system responding?

In [0]:
%sql
SELECT 
    ROUND(AVG(response_time_ms), 0) AS avg_ms,
    ROUND(MIN(response_time_ms), 0) AS min_ms,
    ROUND(MAX(response_time_ms), 0) AS max_ms,
    ROUND(PERCENTILE(response_time_ms, 0.5), 0) AS p50_ms,
    ROUND(PERCENTILE(response_time_ms, 0.95), 0) AS p95_ms,
    COUNT(*) AS total_queries
FROM workspace.default.query_logs
WHERE response_time_ms > 0

## 4. Response Time by Domain
Which domains take longer to answer?

In [0]:
%sql
SELECT 
    domain_detected AS legal_domain,
    COUNT(*) AS queries,
    ROUND(AVG(response_time_ms), 0) AS avg_ms,
    ROUND(PERCENTILE(response_time_ms, 0.95), 0) AS p95_ms
FROM workspace.default.query_logs
WHERE domain_detected IS NOT NULL AND response_time_ms > 0
GROUP BY domain_detected
ORDER BY avg_ms DESC

## 5. Most Frequently Cited Sections
Which BNS sections / Articles are cited most in responses?

In [0]:
from pyspark.sql import functions as F

logs_df = spark.table(TABLE).filter(F.col("sections_cited").isNotNull())

# Parse the JSON array of sections
cited_df = (
    logs_df
    .select(F.explode(F.from_json(F.col("sections_cited"), "ARRAY<STRING>")).alias("section"))
    .groupBy("section")
    .agg(F.count("*").alias("citation_count"))
    .orderBy(F.desc("citation_count"))
    .limit(20)
)

display(cited_df)

## 6. Query Volume Over Time

In [0]:
%sql
SELECT 
    DATE(timestamp) AS query_date,
    COUNT(*) AS query_count,
    COUNT(DISTINCT user_lang) AS unique_languages,
    COUNT(DISTINCT domain_detected) AS unique_domains
FROM workspace.default.query_logs
GROUP BY DATE(timestamp)
ORDER BY query_date


## 7. Model Usage

In [0]:

# Code
%sql
SELECT 
    model_used,
    retrieval_backend,
    COUNT(*) AS query_count,
    ROUND(AVG(response_time_ms), 0) AS avg_ms
FROM workspace.default.query_logs
GROUP BY model_used, retrieval_backend
ORDER BY query_count DESC


## 8. Recent Queries (sample)

In [0]:
# Code
%sql
SELECT 
    timestamp,
    user_lang,
    SUBSTRING(query_text, 1, 80) AS query_preview,
    domain_detected,
    response_time_ms
FROM workspace.default.query_logs
ORDER BY timestamp DESC
LIMIT 20


## 9. Summary Statistics

In [0]:

# Code
%sql
SELECT
    COUNT(*) AS total_queries,
    COUNT(DISTINCT user_lang) AS languages_used,
    COUNT(DISTINCT domain_detected) AS domains_covered,
    ROUND(AVG(response_time_ms) / 1000, 1) AS avg_response_sec,
    MIN(timestamp) AS first_query,
    MAX(timestamp) AS last_query
FROM workspace.default.query_logs